In [1]:
from pathlib import Path

Path("data/cache_opensky").mkdir(parents=True, exist_ok=True)
Path("data/outputs").mkdir(parents=True, exist_ok=True)


In [3]:
import json
import requests
from pathlib import Path

# Load credentials
creds = json.loads(Path("credentials.json").read_text())
CLIENT_ID = creds["clientId"]
CLIENT_SECRET = creds["clientSecret"]

# Correct token endpoint (per OpenSky docs)
TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"

token_resp = requests.post(
    TOKEN_URL,
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    },
    timeout=30,
)

if token_resp.status_code != 200:
    raise RuntimeError(f"Token request failed: {token_resp.status_code} {token_resp.text[:500]}")

token = token_resp.json()["access_token"]
headers = {"Authorization": f"Bearer {token}"}

print("Got access token ✔")


Got access token ✔


In [19]:
import pandas as pd
from datetime import datetime, timedelta, timezone, date
from pathlib import Path
import json
import requests

AIRPORT_ICAO = "EKCH"
BASE_URL = "https://opensky-network.org/api"

CACHE_DIR = Path("data/cache_opensky")
OUT_DIR = Path("data/outputs")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

def cached_get_json(url, params, cache_path):
    if cache_path.exists():
        return json.loads(cache_path.read_text())
    r = requests.get(url, params=params, headers=headers, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"Request failed {r.status_code}: {r.text[:500]}")
    data = r.json()
    cache_path.write_text(json.dumps(data))
    return data

def tidy_flights(records, kind):
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame.from_records(records).copy()
    df["kind"] = kind

    keep = ["kind","icao24","callsign","firstSeen","lastSeen","estDepartureAirport","estArrivalAirport"]
    df = df[[c for c in keep if c in df.columns]]

    df["callsign"] = df["callsign"].astype(str).str.strip().replace({"None": None})
    df["firstSeen_utc"] = pd.to_datetime(df["firstSeen"], unit="s", utc=True)
    df["lastSeen_utc"]  = pd.to_datetime(df["lastSeen"],  unit="s", utc=True)
    df["date_utc"] = df["firstSeen_utc"].dt.date
    df["flight_key"] = df["icao24"].astype(str) + "_" + df["firstSeen"].astype(str)
    df["obs_duration_min"] = (df["lastSeen_utc"] - df["firstSeen_utc"]).dt.total_seconds() / 60.0

    # callsign prefix as a simple "carrier proxy"
    df["callsign_prefix"] = df["callsign"].str.extract(r"^([A-Z]{2,3})", expand=False)

    return df

def fetch_airport_month_utc(year: int, month: int):
    start = datetime(year, month, 1, 0, 0, 0, tzinfo=timezone.utc)
    if month == 12:
        end = datetime(year+1, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
    else:
        end = datetime(year, month+1, 1, 0, 0, 0, tzinfo=timezone.utc)

    frames = []
    day = start
    while day < end:
        day_end = day + timedelta(days=1)
        b = int(day.timestamp())
        e = int(day_end.timestamp())

        arr_cache = CACHE_DIR / f"arrivals_{AIRPORT_ICAO}_{b}_{e}.json"
        dep_cache = CACHE_DIR / f"departures_{AIRPORT_ICAO}_{b}_{e}.json"

        arrivals = cached_get_json(
            f"{BASE_URL}/flights/arrival",
            {"airport": AIRPORT_ICAO, "begin": b, "end": e},
            arr_cache
        )
        departures = cached_get_json(
            f"{BASE_URL}/flights/departure",
            {"airport": AIRPORT_ICAO, "begin": b, "end": e},
            dep_cache
        )

        df_day = pd.concat([tidy_flights(arrivals, "arrival"),
                            tidy_flights(departures, "departure")],
                           ignore_index=True)

        print(f"{day.date()} UTC | rows: {len(df_day)}")
        frames.append(df_day)

        day = day_end

    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not df.empty:
        df = df.drop_duplicates(subset=["flight_key", "kind"])
    return df

df_jan_2025 = fetch_airport_month_utc(2025, 1)
print("Total rows (Jan 2025):", len(df_jan_2025))

out = OUT_DIR / "ekch_flights_2025_01.csv"
df_jan_2025.to_csv(out, index=False)
out


2025-01-01 UTC | rows: 399
2025-01-02 UTC | rows: 565
2025-01-03 UTC | rows: 580
2025-01-04 UTC | rows: 450
2025-01-05 UTC | rows: 536
2025-01-06 UTC | rows: 575
2025-01-07 UTC | rows: 467
2025-01-08 UTC | rows: 450
2025-01-09 UTC | rows: 511
2025-01-10 UTC | rows: 557
2025-01-11 UTC | rows: 395
2025-01-12 UTC | rows: 513
2025-01-13 UTC | rows: 552
2025-01-14 UTC | rows: 448
2025-01-15 UTC | rows: 454
2025-01-16 UTC | rows: 511
2025-01-17 UTC | rows: 562
2025-01-18 UTC | rows: 406
2025-01-19 UTC | rows: 505
2025-01-20 UTC | rows: 554
2025-01-21 UTC | rows: 446
2025-01-22 UTC | rows: 475
2025-01-23 UTC | rows: 533
2025-01-24 UTC | rows: 567
2025-01-25 UTC | rows: 422
2025-01-26 UTC | rows: 530
2025-01-27 UTC | rows: 573
2025-01-28 UTC | rows: 473
2025-01-29 UTC | rows: 476
2025-01-30 UTC | rows: 540
2025-01-31 UTC | rows: 577
Total rows (Jan 2025): 15602


PosixPath('data/outputs/ekch_flights_2025_01.csv')

In [20]:
df_jan_2025.head()

,kind,icao24,callsign,firstSeen,lastSeen,estDepartureAirport,estArrivalAirport,firstSeen_utc,lastSeen_utc,date_utc,flight_key,obs_duration_min,callsign_prefix
0,arrival,4952c3,TAP56RM,1735759231,1735770592,LPPT,EKCH,2025-01-01 19:20:31+00:00,2025-01-01 22:29:52+00:00,2025-01-01,4952c3_1735759231,189.350000,TAP
1,arrival,3c6743,DLH832,1735766511,1735770268,EDDF,EKCH,2025-01-01 21:21:51+00:00,2025-01-01 22:24:28+00:00,2025-01-01,3c6743_1735766511,62.616667,DLH
2,arrival,45ab54,NSZ3TW,1735758172,1735769619,LEMG,EKCH,2025-01-01 19:02:52+00:00,2025-01-01 22:13:39+00:00,2025-01-01,45ab54_1735758172,190.783333,NSZ
3,arrival,471efb,WZZ99JN,1735762028,1735769086,LYBE,EKCH,2025-01-01 20:07:08+00:00,2025-01-01 22:04:46+00:00,2025-01-01,471efb_1735762028,117.633333,WZZ
4,arrival,39bda8,AFR44SC,1735763634,1735768504,LFPG,EKCH,2025-01-01 20:33:54+00:00,2025-01-01 21:55:04+00:00,2025-01-01,39bda8_1735763634,81.166667,AFR


In [ ]:
daily_counts = df_jan_2025.groupby(["date_utc", "kind"]).size().unstack(fill_value=0)
daily_counts


kind,arrival,departure
date_utc,,
2025-01-01,204,195
2025-01-02,291,274
2025-01-03,305,275
2025-01-04,229,221
2025-01-05,276,260
2025-01-06,297,278
2025-01-07,244,223
2025-01-08,235,215
2025-01-09,266,245
